## the libraries for rl 

In [1]:
import math as m
import numpy as np
import random 
import stable_baselines3 as sb3
#import gym 
import gymnasium as gym
import gymnasium_robotics
import torch
import torch.nn as nn
import torch.nn.functional as F
import random 

from abc import ABC, abstractmethod



Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
AdroitHandRelocateDense-v1, AdroitHandHammerDense-v1, AdroitHandDoorDense-v1 environment's reward functions were updated in v1.2.1 without an environment version update. Therefore, use gymnasium-robotics==1.2.0 for v1 reproducibility or use v2 in gymnasium-robotics>=1.4.3. See https://github.com/Farama-Foundation/Gymnasium-Robotics/pull/220 for more details


In [2]:
import torch
import torch.nn as nn

print(torch.__version__)

net = nn.Linear(4, 4)
#optimizer = torch.optim.SGD(net.parameters(), lr=3e-4)

print("works")

2.13.0+cu130
works


## all the tools needed fro RL 

### gae (generalized advantages estimation) calculation

In [3]:
def compute_gae(deltas, gamma=0.99, lam=0.95):
    """Compute GAE advantages from a list of TD residuals (deltas)"""
    advantages = torch.zeros(len(deltas))
    gae = torch.tensor(0.0)
    
    # Backward pass - this is the key
    for t in reversed(range(len(deltas))):
        gae = deltas[t] + gamma * lam * gae
        advantages[t] = gae
    
    return advantages


In [4]:
# Simulate adding steps one by one (as in your question)
deltas = []

print("Incremental GAE Update:\n")

for step in range(1, 7):
    # Simulate receiving a new step
    new_delta = round(np.random.uniform(-1.0, 2.0), 3)   # random for demo
    deltas.append(new_delta)
    
    print(f"After step {step} (new δ = {new_delta}):")
    
    # Recompute GAE using ALL data available so far
    advantages = compute_gae(deltas, gamma=0.99, lam=0.95)
    
    for i, adv in enumerate(advantages):
        print(f"   A[{i+1:2d}] = {adv:.3f}")
    print("-" * 50)

Incremental GAE Update:

After step 1 (new δ = 0.327):
   A[ 1] = 0.327
--------------------------------------------------
After step 2 (new δ = -0.447):
   A[ 1] = -0.093
   A[ 2] = -0.447
--------------------------------------------------
After step 3 (new δ = -0.127):
   A[ 1] = -0.206
   A[ 2] = -0.566
   A[ 3] = -0.127
--------------------------------------------------
After step 4 (new δ = 1.405):
   A[ 1] = 0.963
   A[ 2] = 0.676
   A[ 3] = 1.194
   A[ 4] = 1.405
--------------------------------------------------
After step 5 (new δ = 0.197):
   A[ 1] = 1.117
   A[ 2] = 0.840
   A[ 3] = 1.369
   A[ 4] = 1.590
   A[ 5] = 0.197
--------------------------------------------------
After step 6 (new δ = 0.317):
   A[ 1] = 1.350
   A[ 2] = 1.088
   A[ 3] = 1.632
   A[ 4] = 1.871
   A[ 5] = 0.495
   A[ 6] = 0.317
--------------------------------------------------


### net

In [5]:
# Uncomment if you want to use this simple version Brick 1 and 2 

# ── BRICK 1: reusable block (your Block, kept exactly) ──────────────────
class Block(nn.Module):
    """A reusable residual-style chunk: linear → norm → activation"""
    def __init__(self, dim, activation=nn.ReLU):
        super().__init__()
        self.fc   = nn.Linear(dim, dim)
        self.norm = nn.LayerNorm(dim)
        self.act  = activation()

    def forward(self, x):
        return self.act(self.norm(self.fc(x)))

# ── BRICK 2: shared body ─────────────────────────────────────────────────
class SharedBody(nn.Module):
    """
    Encodes the state φ(s) into a shared representation.
    Both actor and critic read from this.
    """
    def __init__(
        self,
        n_features: int,
        hidden_dim: int = 64,
        n_layers: int = 2,
        activation=nn.ReLU
    ):
        super().__init__()

        self.input_layer = nn.Linear(n_features, hidden_dim)

        layers = [
            self.input_layer,
            activation()
        ]

        for i in range(1, n_layers + 1):
            setattr(self, f"block{i}", Block(hidden_dim, activation))

        for i in range(1, n_layers + 1):
            layers.append(getattr(self, f"block{i}"))

        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)


# class SharedBody(nn.Module): # efficient version
#     def __init__(
#         self,
#         n_layers: int,
#         n_features: int,
#         hidden_dim: int = 64,
#         activation=nn.ReLU
#     ):
#         super().__init__()

#         self.network = nn.Sequential(
#             nn.Linear(n_features, hidden_dim),
#             activation(),
#             *[Block(hidden_dim, activation) for _ in range(n_layers)]
#         )

        
#vars() was useful here 

In [6]:

# ── BRICK 3: actor head ──────────────────────────────────────────────────
class ActorHead(nn.Module):
    """
    Takes shared body output → outputs a probability distribution over actions.
    π_θ(a|s) = softmax(W · h + b)
    """
    def __init__(self, hidden_dim: int, action_dim: int):
        super().__init__()
        self.head = nn.Linear(hidden_dim, action_dim)

    def forward(self, h):
        return F.softmax(self.head(h), dim=-1)   # shape: (action_dim,)

# added class for predicting and choosing the action in a continuous env
class GaussianActor(nn.Module):
    """
    Takes shared body output → outputs a Gaussian probability distribution over continuous actions.
    μ = W · h + b
    σ = exp(log_std)

    π_θ(a|s) = Normal(μ, σ)
    """
    def __init__(self, hidden_dim: int, action_dim: int):
            super().__init__()

            self.mean = nn.Linear(hidden_dim,action_dim)
            self.log_std = nn.Parameter(torch.zeros(action_dim))
    
    def forward(self, h):
        mean = self.mean(h)
        std = torch.exp(self.log_std)

        return torch.distributions.Normal(mean,std)


# ── BRICK 4: critic head ─────────────────────────────────────────────────
class CriticHead(nn.Module):
    """
    Takes shared body output → outputs a single scalar V(s).
    No activation — value can be any real number.
    """
    def __init__(self, hidden_dim: int):
        super().__init__()
        self.head = nn.Linear(hidden_dim, 1)

    def forward(self, h):
        return self.head(h).squeeze(-1)           # shape: scalar


# ── BRICK 5: full Actor-Critic network ───────────────────────────────────
class ActorCritic(nn.Module):
    """
    One network, two outputs:
        actor  → π_θ(a|s)   used for: action selection + policy loss
        critic → V_θ(s)     used for: GAE computation  + value loss
    """
    def __init__(self, 
                n_features: int,
                action_dim: int, 
                hidden_dim: int = 64,
                n_layers = 2,
                action_mode = "discrete"):
        super().__init__()
        self.action_mode = action_mode
        self.body   = SharedBody(n_features, hidden_dim,n_layers)
        self.critic = CriticHead(hidden_dim)
        if action_mode == "discrete":
            self.actor  = ActorHead(hidden_dim, action_dim)
        elif (action_mode== "continuous") or (action_mode == "Gauss"):
            self.actor  = GaussianActor(hidden_dim, action_dim)

        else: 
            raise ValueError(
                "actor_mode must be 'discrete' or 'continous'|'Gauss'"
            )

        self._init_weights()

    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.orthogonal_(m.weight, gain=np.sqrt(2))
                nn.init.constant_(m.bias, 0.0)
        # small gain → initial policy near-uniform → more exploration
        nn.init.orthogonal_(self.critic.head.weight, gain=1.0)
        nn.init.constant_(self.critic.head.bias, 0.0)

        if self.action_mode == "discrete":

            nn.init.orthogonal_(self.actor.head.weight,gain=0.01)
            nn.init.constant_(self.actor.head.bias,0.0)
        else: # the case where gauß is needed
            nn.init.orthogonal_(self.actor.mean.weight,gain=0.01)
            nn.init.constant_(self.actor.mean.bias,0.0)


    def forward(self, x):
        h     = self.body(x)
        value = self.critic(h)   # V_θ(s)
        #probs = self.actor(h)    # π_θ(a|s)

        if self.action_mode == "discrete":

            probs = self.actor(h)    # π_θ(a|s)
            dist = torch.distributions.Categorical(probs)
        else : 
            dist = self.actor(h)    # π_θ(a|s)
        
        return dist, value

    def get_action(self, state_vec: np.ndarray):
        device=next(self.parameters()).device
        x = torch.tensor(state_vec,dtype=torch.float32, device = device)
        dist, value = self.forward(x)
        action = dist.sample()
        log_prob = dist.log_prob(action)

        if self.action_mode == "continuous":
            log_prob = log_prob.sum()

        if self.action_mode == "discrete":
            action = action.item()

        else:
            action = action.detach().cpu().numpy()

        return action, log_prob, value

# ── BRICK 6: your custom loss (kept from your code) ──────────────────────
class MyLoss(nn.Module):
    """MSE loss — used for critic: (V(s) - y_t)²"""
    def forward(self, y_pred, y_true):
        return ((y_pred - y_true) ** 2).mean()
    

### loss functions

In [7]:
class Loss(nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self, *args, **kwargs):
        raise NotImplementedError


In [8]:
class Loss_Entropy(Loss):
    """Entropy encourages exploration."""
    def forward(self, dist) -> torch.Tensor:
        return dist.entropy().mean()


class Loss_Critic(Loss):
    """L_V = (V(s) - y)²"""
    def forward(
        self,
        v_pred: torch.Tensor,
        v_target: torch.Tensor
    ) -> torch.Tensor:

        return F.mse_loss(v_pred, v_target)


class Loss_Actor(Loss):
    """L_pi = -ρ·A — PPO clipped policy gradient"""

    def __init__(self, clip_epsilon=0.2):
        super().__init__()
        self.eps = clip_epsilon

    def forward(
        self,
        log_prob_new: torch.Tensor,
        log_prob_old: torch.Tensor,
        advantage: torch.Tensor
    ) -> torch.Tensor:

        # ρ = π_new / π_old
        rho = torch.exp(log_prob_new - log_prob_old)

        clipped = torch.clamp(
            rho,
            1 - self.eps,
            1 + self.eps
        )

        # PPO clipped objective
        L_clip = torch.min(
            rho * advantage,
            clipped * advantage
        )

        return -L_clip.mean()


class Loss_PPO(Loss):

    def __init__(self, cv=0.5, ce=0.01, clip_epsilon=0.2):
        super().__init__()

        self.cv = cv
        self.ce = ce

        self.L_pi = Loss_Actor(clip_epsilon)
        self.L_v = Loss_Critic()

    def forward(
        self,
        log_prob_new,
        log_prob_old,
        advantage,
        v_pred,
        v_target,
        entropy
    ):
        actor_loss = self.L_pi(
            log_prob_new,
            log_prob_old,
            advantage
        )

        critic_loss = self.L_v(
            v_pred,
            v_target
        )

        return (
            actor_loss
            + self.cv * critic_loss
            - self.ce * entropy
        )

### learning strategy (for improvement and update)

In [9]:
class LearningStrategy(ABC):
    @abstractmethod
    def update(self, policy, experience): pass
    @abstractmethod
    def snapshot(self): pass

In [10]:
class PPO(LearningStrategy):
    def __init__(
        self,
        optimizer,
        clip_epsilon: float = 0.2,
        cv: float = 0.5,
        ce: float = 0.01,
        gamma: float = 0.99,
        lam: float = 0.95,
        n_epochs: int = 1
    ):
        self.optimizer = optimizer
        self.clip_epsilon = clip_epsilon
        self.cv = cv
        self.ce = ce
        self.gamma = gamma
        self.lam = lam
        self.n_epochs = n_epochs
        self.loss_fn = Loss_PPO(cv=cv, ce=ce, clip_epsilon=clip_epsilon)

    def compute_gae(self, deltas, dones, gamma=None, lam=None):
        gamma = gamma if gamma is not None else self.gamma
        lam = lam if lam is not None else self.lam
        T = len(deltas)

        device = deltas[0].device

        advantages = torch.zeros(T,device = device)
        gae = torch.tensor(0.0,device = device)

        for t in reversed(range(T)):
            if dones[t]:
                gae = torch.tensor(0.0,device = device)
            gae = deltas[t] + gamma * lam * gae
            advantages[t] = gae.detach()
        return advantages

    def update(self, policy, rollout):
        if policy.net.action_mode == "discrete":
            return self._update_discrete(policy, rollout)
        elif policy.net.action_mode == "continuous":
            return self._update_continuous(policy, rollout)
        else:
            raise ValueError(f"Unknown action mode: {policy.net.action_mode}")

    def _update_discrete(self, policy, rollout):
        return self._update_common(policy, rollout, continuous=False)

    def _update_continuous(self, policy, rollout):
        return self._update_common(policy, rollout, continuous=True)

    def _update_common(self, policy, rollout, continuous: bool):

        device = next(policy.net.parameters()).device

        deltas = rollout["deltas"]
        states_visited = rollout["states_visited"]
        log_probs_old = rollout["log_probs_old"]
        actions_taken = rollout["actions_taken"]
        values_visited = rollout["values_visited"]
        dones = rollout["dones"]

        # ----- Advantages & Returns -----
        A = self.compute_gae(deltas, dones)
        v_tensor = torch.stack([v.detach().to(device) for v in values_visited])
        returns = (A + v_tensor).detach()

        # Normalise advantages
        if torch.isfinite(A.std()) and A.std() > 1e-8:
            A = (A - A.mean()) / (A.std() + 1e-8)
        else:
            A = A - A.mean()

        total_loss = 0.0

        for _ in range(self.n_epochs):
            log_probs_new = []
            values_new = []
            entropies = []

            for s, a in zip(states_visited, actions_taken):
                #MODIFIED
                #x = torch.as_tensor(policy.state_to_vector_fn(s),dtype=torch.float32,device=device)
                x = torch.as_tensor(s,dtype=torch.float32,device=device)
                dist, value = policy.net(x)

                if continuous:
                    #action = torch.as_tensor(a, dtype=torch.float32,device=device)
                    log_prob = dist.log_prob(action).sum()
                    entropy = dist.entropy().sum()
                else:
                    #MODIFIED
                    #action_idx = policy.actions.index(a)
                    action_idx = torch.as_tensor(
                        a,
                        dtype=torch.long,
                        device=device
                        )
                    log_prob = dist.log_prob(torch.tensor(action_idx,device = device))
                    entropy = dist.entropy()

                log_probs_new.append(log_prob)
                values_new.append(value)
                entropies.append(entropy)

            log_probs_new = torch.stack(log_probs_new)
            log_probs_old_t = torch.stack([ 
                lp.to(device) for lp in log_probs_old
                ])
            v_pred = torch.stack(values_new)
            entropy = torch.stack(entropies).mean()

            loss = self.loss_fn(
                log_probs_new,
                log_probs_old_t,
                A,
                v_pred,
                returns,
                entropy
            )

            self.optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(policy.net.parameters(), max_norm=0.5)
            self.optimizer.step()

            total_loss += loss.item()

        return total_loss / self.n_epochs

    def snapshot(self):
        return {
            "gamma": self.gamma,
            "lam": self.lam,
            "clip_epsilon": self.clip_epsilon,
            "cv": self.cv,
            "ce": self.ce,
            "lr": self.optimizer.param_groups[0]["lr"],
        }

### policy

In [11]:
from abc import ABC, abstractmethod

class Policy(ABC):
    @abstractmethod
    def select_action(self, state): pass
    @abstractmethod
    def action_distribution(self, state): pass
    @abstractmethod
    def snapshot(self): pass

In [12]:
def s2v(state):
    return state

class Policy_PPO(Policy):

    def __init__(self, network, actions=None,
                 state_to_vector_fn= lambda x: x):

        super().__init__()

        self.pi = {}
        self.net = network
        self.actions = actions
        self.state_to_vector_fn = state_to_vector_fn

    def select_action(self, state):

        x = self.state_to_vector_fn(state)

        action, log_prob, value = self.net.get_action(x)

        # Only discrete actions need index → environment action conversion
        if self.net.action_mode == "discrete":
            action = self.actions[action]

        return action, log_prob, value

    def action_distribution(self, state):

        x = torch.tensor(
            self.state_to_vector_fn(state),
            dtype=torch.float32
        )

        with torch.no_grad():
            dist, _ = self.net(x)

        return dist

    def snapshot(self):
        return self.net.state_dict()

### agent

In [13]:
# ════════════════════════════════════════════════════════════════════
#  BRICK 4 — AGENT
#  The self. Owns the policy (permanent). Borrows the strategy (swappable).
#  Contains no learning logic — purely coordinates the other bricks.
# ════════════════════════════════════════════════════════════════════
    
class Agent:

    def __init__(self, strategy: LearningStrategy, policy: Policy):
        self.policy   = policy    # permanent — never replaced
        self.strategy = strategy  # swappable — plug any algorithm in
        self.memory   = []        # full experience trace

    def act(self, state) -> str:
        """Ask the strategy what to do, passing the policy as context."""
        return self.policy.select_action(state)

    def learn(self, experience:dict): # experience = {"state": "A","action": "right","reward": 1,"next_state": "B","done": False}
        """Tell the strategy what happened; it writes into the policy."""
        

        self.strategy.update(self.policy, experience)
        self.memory.append((
            experience["state"].name, 
            experience["action"], 
            experience["reward"], 
            experience["next_state"].name, 
            experience["done"]
        ))

    def swap_strategy(self, new_strategy: LearningStrategy):
        """
        Replace the learning algorithm.
        The policy — and everything it has learned — is untouched.
        """
        self.strategy = new_strategy

In [14]:

class Agent_PPO(Agent):

    def __init__(self, strategy: PPO, policy: Policy_PPO):
        super().__init__(strategy, policy)

    def act(self, state):
        return self.policy.select_action(state)

    def learn(self, rollout):
        loss = self.strategy.update(self.policy, rollout)
        self.memory.append({
            "rollout_size": len(rollout["deltas"]), 
            "loss":loss
        })

    def value(self, state,device="cpu"):
        x = torch.as_tensor(
            self.policy.state_to_vector_fn(state),
            dtype=torch.float32,device=device
        )

        _, value = self.policy.net(x)
        return value

## here is the env from mujoco

In [15]:
#env = gym.make("CartPole-v1")
env = gym.make("InvertedPendulum-v5")#,render_mode = "human")
print("observation space",env.observation_space.shape[0])
print("action space",env.action_space.shape[0])
print("reward",)

observation space 4
action space 1
reward


In [16]:
# Here as a dashboard for developping the customable reward

import inspect
import gymnasium as gym
from gymnasium import spaces
import numpy as np

### custom obs
class CustomObservation(gym.ObservationWrapper):

    def __init__(self, env, s2v):

        super().__init__(env)

        self.s2v = s2v

        # Determine size of the new observation
        example = self.s2v(env.observation_space.sample())

        self.observation_space = spaces.Box(
            low=-np.inf,
            high=np.inf,
            shape=example.shape,
            dtype=np.float32
        )

    def observation(self, obs):

        return self.s2v(obs).astype(np.float32)

### custom reward

class RewardCustom(gym.Wrapper):
    """
    reward_fn can take any subset of these argument names (or **kwargs for all):
    obs, action, next_obs, reward, terminated, truncated, info, env
    """
    def __init__(self, env, reward_fn):
        super().__init__(env)
        self.reward_fn = reward_fn
        params = inspect.signature(reward_fn).parameters
        self._takes_all = any(p.kind == p.VAR_KEYWORD for p in params.values())
        self._params = set(params)
        self._obs = None

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        self._obs = obs
        return obs, info

    def step(self, action):

        state = self.env.unwrapped.state

        next_obs, reward, terminated, truncated, info = self.env.step(action)

        next_state = self.env.unwrapped.state

        ctx = dict(
            state=state,
            action=action,
            next_state=next_state,
            obs=self._obs,
            next_obs=next_obs,
            reward=reward,
            terminated=terminated,
            truncated=truncated,
            info=info,
            env=self.env.unwrapped,
        )

        kwargs = (
            ctx if self._takes_all
            else {k: v for k, v in ctx.items() if k in self._params}
        )

        info["original_reward"] = reward

        new_reward = float(self.reward_fn(**kwargs))

        self._obs = next_obs

        return next_obs, new_reward, terminated, truncated, info

In [17]:
# Here is to design the custom reward / obs functiond

## obs func

def s2v_fn(obs,*args,**kwargs):
    pass

## reward func

def reward_func(*args,**kwargs):
    pass

In [18]:

#env = gym.make("CartPole-v1")
inv_env = gym.make("InvertedPendulum-v5",)#render_mode = "human")
#print(env.action_space)

# space to plug in the reward function and observation function but how to ?

# env = ObservationCustom(env, s2v_fn=my_s2v) # the function s2v is designed on a specific cell 
# env = RewardCustom(env,reward_fn = my_reward) # the function reward is designed on a specific cell

# ========================================================= 
#  PPO AGENT 
# ========================================================= 

# Choose device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = torch.device("cpu")   # force CPU
# device = torch.device("cuda")  # force CUDA



print("Using ", device)

n_features = inv_env.observation_space.shape[0] #env.observation_space.shape[0] 
action_dim = inv_env.action_space.shape[0] #env.action_space.shape[0]
net = ActorCritic( n_features=n_features, 
                  action_dim=action_dim, 
                  hidden_dim=64 ,
                  action_mode="continuous"
                  ).to(device)

policy = Policy_PPO(net,state_to_vector_fn= lambda x:x ) 
optimizer = torch.optim.Adam( net.parameters(), lr=3e-4 ) 
ppo = PPO( optimizer=optimizer, clip_epsilon=0.2, gamma=0.9, lam=0.95, cv=0.5, ce=0.01, n_epochs=10 ) 
agent_gym = Agent_PPO( strategy=ppo, policy=policy )

Using  cuda


In [21]:
N = 10
steps = 10
for episode in range(1,N+1):
    state, info = inv_env.reset()

    print(f"info start from reset of episode {episode} : {info}")

    done = False
    score = 0

    while not done:
        action, log_prob, value = agent_gym.act(state)

        obs, reward, terminated, truncated, info = inv_env.step(action)

        state = obs
        done = terminated or truncated

        score += reward

        print(
            f"action: {action}\n"
            f"obs: {obs}\n"
            f"reward: {reward}\n"
            f"terminated?: {terminated}\n"
            f"truncated?: {truncated}\n"
            f"info: {info}\n"
        )
    inv_env.close()
    print(f"Total Score: {score}")

info start from reset of episode 1 : {}
action: [1.0664551]
obs: [ 0.0051001  -0.02360726  0.3485792  -0.8162332 ]
reward: 1
terminated?: False
truncated?: False
info: {'reward_survive': 1}

action: [0.44039437]
obs: [ 0.02189479 -0.06243287  0.49134434 -1.12811939]
reward: 1
terminated?: False
truncated?: False
info: {'reward_survive': 1}

action: [1.8997864]
obs: [ 0.05410372 -0.13644756  1.1172247  -2.56289518]
reward: 1
terminated?: False
truncated?: False
info: {'reward_survive': 1}

action: [0.46750054]
obs: [ 0.10167008 -0.2454151   1.26136836 -2.89792118]
reward: 0
terminated?: True
truncated?: False
info: {'reward_survive': 0}

Total Score: 3
info start from reset of episode 2 : {}
action: [-0.9309313]
obs: [-0.00770534  0.01754051 -0.31508563  0.70992117]
reward: 1
terminated?: False
truncated?: False
info: {'reward_survive': 1}

action: [-0.32319248]
obs: [-0.02239136  0.05039459 -0.41945495  0.93599244]
reward: 1
terminated?: False
truncated?: False
info: {'reward_survive':